# Purpose of the module: 

Read the basic raw data files and store them as fast available file in the current working directory. 

!!! Attention !!! If the raw data file "X" is loaded this lasts about 15 minutes.

In [2]:
#
# Cell 1: Libraries laden
#

import pandas as pd
import numpy as np
import wfdb
import ast
import pickle

In [3]:
#
# Cell 2:Path for input data and definition of sampling rate
#

#path = 'C:/Users/Buchh/OneDrive/Leuphana_Data_Science/Masterarbeit/ECG/Python/Daten_einlesen/test_data/'
path = 'D:/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/'
sampling_rate=100


In [4]:
#
# Cell 3: Definition of functions
#

def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data])
    return data

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

def aggregate_form_rhythm(y_dic): #BuK
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].form_rhythm)
    return list(set(tmp))


In [8]:
#
# Cell 4: Load Y raw data
#

# load and convert annotation data
Y = pd.read_csv(path+'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))

#display (Y.scp_codes)
Y.reset_index(level=0,inplace = True)
display (Y.scp_codes)

0                 {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}
1                             {'NORM': 80.0, 'SBRAD': 0.0}
2                               {'NORM': 100.0, 'SR': 0.0}
3                               {'NORM': 100.0, 'SR': 0.0}
4                               {'NORM': 100.0, 'SR': 0.0}
                               ...                        
21794    {'NDT': 100.0, 'PVC': 100.0, 'VCLVH': 0.0, 'ST...
21795             {'NORM': 100.0, 'ABQRS': 0.0, 'SR': 0.0}
21796                           {'ISCAS': 50.0, 'SR': 0.0}
21797                           {'NORM': 100.0, 'SR': 0.0}
21798                           {'NORM': 100.0, 'SR': 0.0}
Name: scp_codes, Length: 21799, dtype: object

In [10]:
Y.scp_codes[150:200]

150                           {'NORM': 100.0, 'SR': 0.0}
151                            {'NORM': 80.0, 'SR': 0.0}
152                           {'NST_': 100.0, 'SR': 0.0}
153                             {'IMI': 35.0, 'SR': 0.0}
154    {'ASMI': 50.0, 'LAFB': 100.0, 'LVH': 50.0, 'IS...
155                         {'NORM': 80.0, 'SARRH': 0.0}
156                                     {'ISCAL': 100.0}
157                           {'NORM': 100.0, 'SR': 0.0}
158                            {'NORM': 80.0, 'SR': 0.0}
159                          {'ISCAL': 100.0, 'SR': 0.0}
160                           {'ISCAL': 50.0, 'SR': 0.0}
161                          {'IRBBB': 100.0, 'SR': 0.0}
162             {'NST_': 100.0, 'PVC': 100.0, 'SR': 0.0}
163                          {'ISCIL': 100.0, 'SR': 0.0}
164           {'CRBBB': 100.0, 'LAFB': 100.0, 'SR': 0.0}
165             {'LVH': 100.0, 'ISC_': 100.0, 'SR': 0.0}
166                           {'NORM': 100.0, 'SR': 0.0}
167               {'IMI': 15.0,

In [14]:
#
# Cell 5: Load X raw signal data
# CAUTION: long loading time!!!
#

import time
start = time.time()

X = load_raw_data(Y, sampling_rate, path)

end = time.time()

print('Minutes for loading data:',(end - start)/60)
print('X.shape :',X.shape)



767.8351211547852
X.shape : (21799, 1000, 12)


In [16]:
#
# Cell 6: DIAGNOSTIC SUPERCLASS
# Load scp_statements.csv for diagnostic aggregation
#

agg_df = pd.read_csv(path+'scp_statements.csv', index_col=0)

In [22]:
#
# Cell 7: creates a folder for data storage.
#

from pathlib import Path

# Specify the directory name
directory_path = Path("D:/ECG/X_zero")

# Create the directory
try:
    directory_path.mkdir()
    print(f"Directory '{directory_path}' created successfully.")
except FileExistsError:
    print(f"Directory '{directory_path}' already exists.")
except PermissionError:
    print(f"Permission denied: Unable to create '{directory_path}'.")
except Exception as e:
    print(f"An error occurred: {e}")

Directory 'D:\ECG\x_zero' already exists.


In [24]:
#
# Cell 8: Save X as 'X_np_array.npy' 
#

X = np.asarray(X, dtype=np.float32)
np.save(directory_path/"X_float32.npy", X)
#np.save('X_np_array.npy', X)


In [26]:
#
# Cell 9: Save Y as 'Y_data.pkl' and and agg_df 
# as 'spc_statements.pkl' file
#

with open(directory_path/'Y_data.pkl', 'wb') as f:  # open a text file
    pickle.dump(Y, f) # serialize the list
    f.close()

with open(directory_path/'spc_statements.pkl', 'wb') as f:  # open a text file
    pickle.dump(agg_df, f) # serialize the list
    f.close()

In [ ]:
#
# Cell 10: Load X from 'X_np_array.npy' file
#

X = np.load(directory_path/'X_np_array.npy')


In [23]:
#
# Cell 11: Restore Y from Y_data and agg_df from spc_statements
#

with open(directory_path/'Y_data.pkl', 'rb') as f:

    Y = pickle.load(f) # deserialize using load()
f.close()  

with open(directory_path/'spc_statements.pkl', 'rb') as f:

    agg_df = pickle.load(f) # deserialize using load()
f.close()
